# Extract Hidden States

Runs chr22 through a trained genomic compressor and saves the final transformer layer's hidden representations for downstream SAE interpretability analysis.

**Before running:** upload `model.py`, `data.py`, your checkpoint (`best_model.pt`), and `chr22.npy` to Google Drive (or directly into the Colab session). Adjust the paths in the Configuration cell.

## Environment setup

Enable a GPU runtime: `Runtime → Change runtime type → GPU` (T4, A100, or L4 all work).

In [1]:
!nvidia-smi

Wed Apr 22 03:52:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys

PROJECT_DIR = '/content/drive/MyDrive/genomic_compressor'
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

## Imports

In [4]:
import random
import numpy as np
import torch
from torch.utils.data import DataLoader

from model import DecoderOnlyTransformer
from data import GenomeDataset

## Configuration

In [5]:
CHECKPOINT_PATH = f'{PROJECT_DIR}/checkpoints/best_model.pt'
CHR22_TOKEN_PATH = f'{PROJECT_DIR}/tokenized_data/chr22.npy'
OUTPUT_PATH = f'{PROJECT_DIR}/chr22_hidden_states.pt'

EXTRACT_LAYER = 11
CHUNK_SIZE = 2048
BATCH_SIZE = 16

VOCAB_SIZE = 4096
EMBED_DIM = 512
NUM_HEADS = 8
NUM_LAYERS = 12

SEED = 42

## Setup

In [6]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')
    if device.type == 'cuda':
        print(f'  GPU:    {torch.cuda.get_device_name(0)}')
        print(f'  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    return device


set_seed(SEED)
device = get_device()

Device: cuda
  GPU:    NVIDIA A100-SXM4-80GB
  Memory: 85.1 GB


## Build model and load checkpoint

In [7]:
def build_model(device):
    model = DecoderOnlyTransformer(
        vocab_size=VOCAB_SIZE,
        embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS,
        num_layers=NUM_LAYERS,
        bias=True,
        tie_weights=True,
    ).to(device)

    num_params = sum(p.numel() for p in model.parameters())
    print(f'Model: {NUM_LAYERS}L / {EMBED_DIM}d / {NUM_HEADS}H ({num_params:,} params)')
    return model


def load_checkpoint(path, model, device):
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    bpb = checkpoint.get('val_bpb', None)
    epoch = checkpoint.get('epoch', None)
    print(f'Loaded checkpoint: epoch {epoch}, val bpb {bpb:.4f}' if bpb else f'Loaded checkpoint.')


model = build_model(device)
load_checkpoint(CHECKPOINT_PATH, model, device)

Model: 12L / 512d / 8H (43,072,000 params)
Loaded checkpoint: epoch 4, val bpb 1.4978


## Build data loader

In [8]:
def build_loader(token_path):
    tokens = np.load(token_path)
    chr22_tokens = {'chr22': tokens}

    dataset = GenomeDataset(
        chr22_tokens,
        ['chr22'],
        seq_len=CHUNK_SIZE,
        stride=CHUNK_SIZE,
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,
    )

    print(f'Chr22: {len(tokens):,} tokens → {len(dataset):,} chunks → {len(loader):,} batches')
    return loader


loader = build_loader(CHR22_TOKEN_PATH)

Chr22: 6,526,630 tokens → 3,186 chunks → 200 batches


## Extract hidden states

In [9]:
def extract_hidden_states(model, loader, layer, device):
    model.eval()
    all_hidden_states = []

    with torch.no_grad():
        for batch_idx, (x, _) in enumerate(loader):
            x = x.to(device, non_blocking=True)

            with torch.amp.autocast('cuda', dtype=torch.float16):
                output = model(x, return_hidden_layer=layer)

            all_hidden_states.append(output.hidden_state.cpu())

            if batch_idx % 50 == 0:
                print(f'  Batch {batch_idx:>4} / {len(loader)}')

    hidden_states = torch.cat(all_hidden_states, dim=0).view(-1, EMBED_DIM)
    print(f'  Extracted: {hidden_states.shape}')
    return hidden_states


print(f'Extracting layer {EXTRACT_LAYER} hidden states')
hidden_states = extract_hidden_states(model, loader, EXTRACT_LAYER, device)

Extracting layer 11 hidden states
  Batch    0 / 200
  Batch   50 / 200
  Batch  100 / 200
  Batch  150 / 200
  Extracted: torch.Size([6524928, 512])


## Save to Drive

In [10]:
torch.save(hidden_states, OUTPUT_PATH)
print(f'Saved {tuple(hidden_states.shape)} → {OUTPUT_PATH}')
print(f'Size: {hidden_states.nbytes / 1e9:.2f} GB')

Saved (6524928, 512) → /content/drive/MyDrive/genomic_compressor/chr22_hidden_states.pt
Size: 13.36 GB
